In [1]:
import importlib
import sys
import numpy as np
from collections import Counter
import pandas as pd

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load decision-labeled data
file_path_train = '../../../../../../data/Sepsis/tensor_data/decision_labeled/sepsis_all_5_train.pkl'
sepsis_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/Sepsis/tensor_data/decision_labeled/sepsis_all_5_val.pkl'
sepsis_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# Sepsis dataset dynamic categories, features:
sepsis_all_categories = sepsis_train_dataset.all_categories

sepsis_all_categories_cat = sepsis_all_categories[0]
sepsis_all_categories_num = sepsis_all_categories[1]
for i, cat in enumerate(sepsis_all_categories_cat):
     print(f"Sepsis dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_categories_num):
     print(f"Sepsis dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")
print('\n')
     
# Sepsis dataset static categories, features:
sepsis_all_stat_categories = sepsis_train_dataset.all_static_categories

sepsis_all_stat_categories_cat = sepsis_all_stat_categories[0]
sepsis_all_stat_categories_num = sepsis_all_stat_categories[1]
for i, cat in enumerate(sepsis_all_stat_categories_cat):
     print(f"Sepsis static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(sepsis_all_stat_categories_num):
     print(f"Sepsis static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"Sepsis amount of numerical: {num[1]}")

Sepsis dynamic categorical feature: concept:name, Index position in categorical data list: 0
Sepsis amount of category labels: 18
Sepsis dynamic categorical feature: org:group, Index position in categorical data list: 1
Sepsis amount of category labels: 28
Sepsis dynamic categorical feature: lifecycle:transition, Index position in categorical data list: 2
Sepsis amount of category labels: 3


Sepsis dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: day_in_week, Index position in categorical data list: 2
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: seconds_in_day, Index position in categorical data list: 3
Sepsis amount of numerical: 1
Sepsis dynamic numerical feature: Leucocytes, Index position in categorical data list: 4
Sepsis amount of nu

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['concept:name']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (activity only for T-GAN-LSTM)
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

# Guard tensors are pre-computed and stored in the dataset .pkl files
# (prepared during data loading via prepare_guard_tensors).
print(f"Guard tensors: targets {sepsis_train_dataset._guard_targets.shape}, "
      f"mask {sepsis_train_dataset._guard_mask.shape}, "
      f"confidence {sepsis_train_dataset._guard_confidence.shape}")

Input features encoder:  [['concept:name'], ['case_elapsed_time']]
Features decoder:  [['concept:name'], []]
Guard tensors: targets torch.Size([9750, 50, 18]), mask torch.Size([9750, 50]), confidence torch.Size([9750, 50])


In [5]:
import suffix_pred.models.T_GAN_LSTM
importlib.reload(suffix_pred.models.T_GAN_LSTM)
from suffix_pred.models.T_GAN_LSTM import TaymouriAdversarialLSTM

# Prediction decoder output sequence length (fixed suffix length S)
seq_len_pred = sepsis_train_dataset.min_suffix_size

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# T-GAN-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'concept:name'
activity_feature_index = next(i for i, cat in enumerate(sepsis_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = sepsis_all_categories_cat[activity_feature_index][1]

# concept_name_id: index of the activity feature in the categorical data list
concept_name_id = activity_feature_index

# T-GAN-LSTM model initialization
model = TaymouriAdversarialLSTM(
    data_set_categories=sepsis_all_categories,
    model_feat=model_feat,
    concept_name_id=concept_name_id,
    hidden_size=hidden_size,
    num_layers=num_layers,
    seq_len_pred=seq_len_pred,
    input_size=input_size,
    output_size_act=output_size_act,
    dropout=0.2,
)

/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import TTraining

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Initialize G and D with standard normal distribution (Algorithm 1, step 1)
model.init_weights_normal()

# RMSprop optimizer (Taymouri et al.) with lr=5e-5
learning_rate = 5e-5

# Generator optimizer: RMSprop on encoder-decoder (G)
generator_optimizer = torch.optim.RMSprop(params=model.seq2seq.parameters(), lr=learning_rate)

# Discriminator optimizer: RMSprop on discriminator (D)
discriminator_optimizer = torch.optim.RMSprop(params=model.discriminator.parameters(), lr=learning_rate)

# 100 iterations (Algorithm 1)
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

# Scheduled sampling (same as clean baseline):
# TF ratio anneals from 1.0 → min_teacher_forcing_value via inverse-sigmoid.
# L_guard is applied only at decoder steps that consumed GT input (tf_mask=1).
teacher_forcing_mode = "scheduled"
max_teacher_forcing_value = 1.0
min_teacher_forcing_value = 0.0

# Gumbel-softmax temperature annealing (0.9 - 0)
tau_start = 0.9
tau_min = 0.01

# GAN training mode (Algorithm MLMME)
use_gan = True

optimize_values = {"optimizer": generator_optimizer,
                   "generator_optimizer": generator_optimizer,
                   "discriminator_optimizer": discriminator_optimizer,
                   "scheduler": None,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle,
                   "min_teacher_forcing_value": min_teacher_forcing_value,
                   "max_teacher_forcing_value": max_teacher_forcing_value,
                   "teacher_forcing_mode": teacher_forcing_mode,
                   "use_gan": use_gan,
                   "tau_start": tau_start,
                   "tau_min": tau_min,
                   "beam_width": 3}

# Suffix split value = fixed suffix length S
suffix_data_split_value = sepsis_train_dataset.min_suffix_size
print("Suffix length S:", suffix_data_split_value)
print(f"Teacher forcing config: mode={teacher_forcing_mode}")

# Activity feature EOS id
activity_label_to_id = sepsis_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')
if eos_id is None:
    raise ValueError("Could not find EOS id in activity label mapping.")

# Decision-aware guard regularization weight (λ_g)
lambda_g = 1.0
print(f"Decision-aware training: λ_g = {lambda_g}")

model_output_path = "../../../../../../models/Sepsis/decision/Sepsis_T_GAN_LSTM_v1_DA.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = TTraining(device=device,
                    model=model,
                    data_train=sepsis_train_dataset,
                    data_val=sepsis_val_dataset,
                    optimize_values=optimize_values,
                    suffix_data_split_value=suffix_data_split_value,
                    concept_name_id=concept_name_id,
                    eos_id=eos_id,
                    loss_obj=loss_obj,
                    lambda_g=lambda_g,
                    save_model_n_th_epoch=1,
                    saving_path=model_output_path)

# Train the model (Algorithm 1: MLMME adversarial training + decision-aware guard loss)
train_gen_losses, train_disc_losses, val_losses = trainer.train()

Device: cuda
Suffix length S: 5
Teacher forcing config: mode=scheduled
Decision-aware training: λ_g = 1.0
Device:  cuda
Mode:  GAN (Algorithm 1: MLMME)
Epochs (iterations):  100
Gumbel-softmax τ:  0.9 → 0.01 (exponential anneal)
Teacher forcing mode: scheduled
Scheduled sampling ε: 0.0 -> 1.0 (inverse-sigmoid)


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], LR: None, τ: 0.9000, TF: 1.0000, ε: 0.0000, Gen Loss: 5.2035, Disc Loss: 1.3739
saving model
Epoch [2/100], LR: None, τ: 0.8600, TF: 0.9905, ε: 0.0095, Gen Loss: 5.1564, Disc Loss: 1.3606
saving model
Epoch [3/100], LR: None, τ: 0.8218, TF: 0.9803, ε: 0.0197, Gen Loss: 4.9714, Disc Loss: 1.3480
saving model
Epoch [4/100], LR: None, τ: 0.7853, TF: 0.9692, ε: 0.0308, Gen Loss: 4.5828, Disc Loss: 1.3431
saving model
Epoch [5/100], LR: None, τ: 0.7504, TF: 0.9572, ε: 0.0428, Gen Loss: 4.4730, Disc Loss: 1.3373
saving model
Epoch [6/100], LR: None, τ: 0.7170, TF: 0.9443, ε: 0.0557, Gen Loss: 4.4323, Disc Loss: 1.3269
saving model
Epoch [7/100], LR: None, τ: 0.6852, TF: 0.9305, ε: 0.0695, Gen Loss: 4.4155, Disc Loss: 1.3142
saving model
Epoch [8/100], LR: None, τ: 0.6547, TF: 0.9156, ε: 0.0844, Gen Loss: 4.4253, Disc Loss: 1.2982
saving model
Epoch [9/100], LR: None, τ: 0.6256, TF: 0.8998, ε: 0.1002, Gen Loss: 4.4292, Disc Loss: 1.2757
saving model
Epoch [10/100], LR: None, τ: